[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/29_cnn/01_convolution_from_scratch.ipynb)

# Bài 01 — Tích chập from scratch

**🎯 Mục tiêu**
- Vì sao MLP không hợp với ảnh: số tham số, mất cấu trúc không gian.
- Phép **tích chập 2D** (thực chất là cross-correlation) from scratch: kernel, **stride**, **padding**, công thức kích thước đầu ra; nhiều kênh vào/ra.
- **im2col**: biến tích chập thành nhân ma trận; đối chiếu `torch.nn.functional.conv2d`.
- **Pooling**; **trường tiếp nhận** (receptive field); tính **đồng biến với phép dịch** (translation equivariance).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/29_cnn

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.datasets import SHAPE_CLASSES, make_shapes
from ml_course.visualization import set_style

try:
    import torch
    import torch.nn.functional as F
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()

## 1. Dữ liệu và vấn đề của MLP

Bộ ảnh 28×28 tự sinh (offline): 5 loại hình ở vị trí, kích thước, góc xoay, độ sáng ngẫu nhiên, có nhiễu.

In [ ]:
X, y = make_shapes(2000, seed=0)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, i in zip(axes.ravel(), range(16)):
    ax.imshow(X[i], cmap="gray")
    ax.set_title(SHAPE_CLASSES[y[i]], fontsize=9)
    ax.axis("off")
plt.show()

MLP coi ảnh là vector 784 số:
- Lớp ẩn 256 nơ-ron cần $784 \times 256 \approx 200\,000$ trọng số — với ảnh 224×224×3 là ~38 triệu cho **một** lớp.
- Không biết hai pixel **kề nhau**; một hình tròn ở góc trái và ở góc phải là hai mẫu "khác hẳn" → phải học lại ở **mọi vị trí**.

Ý tưởng tích chập: dùng **một bộ lọc nhỏ** (vd 3×3) trượt khắp ảnh — **chia sẻ trọng số** theo vị trí, chỉ nhìn **vùng lân cận**.

## 2. Tích chập 2D

Với ảnh $X$ và kernel $K$ kích thước $k \times k$:
$$Y[i, j] = \sum_{a=0}^{k-1}\sum_{b=0}^{k-1} X[i\cdot s + a,\; j\cdot s + b]\; K[a, b] + b_0$$
(đây là **cross-correlation**; tích chập toán học lật kernel — trong deep learning không phân biệt vì kernel được học.)

Kích thước đầu ra với padding $p$, stride $s$: $\;H_{out} = \left\lfloor\frac{H + 2p - k}{s}\right\rfloor + 1$.

In [ ]:
def conv2d_naive(x, k, stride=1, padding=0):
    """x: (H, W), k: (kh, kw) — vòng lặp tường minh."""
    if padding:
        x = np.pad(x, padding)
    kh, kw = k.shape
    H_out = (x.shape[0] - kh) // stride + 1
    W_out = (x.shape[1] - kw) // stride + 1
    out = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            patch = x[i * stride:i * stride + kh, j * stride:j * stride + kw]
            out[i, j] = np.sum(patch * k)
    return out


img = X[np.flatnonzero(y == 1)[0]].astype(float)                        # một hình vuông
kernels = {
    "Sobel ngang (cạnh dọc)": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float),
    "Sobel dọc (cạnh ngang)": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float),
    "làm mờ (trung bình 3×3)": np.ones((3, 3)) / 9,
    "Laplacian (mọi cạnh)": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], float),
}
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(img, cmap="gray")
axes[0].set_title("ảnh gốc")
for ax, (name, k) in zip(axes[1:], kernels.items()):
    ax.imshow(conv2d_naive(img, k, padding=1), cmap="RdBu_r")
    ax.set_title(name, fontsize=10)
for ax in axes:
    ax.axis("off")
plt.show()
for (s, p) in [(1, 0), (1, 1), (2, 1), (3, 0)]:
    out = conv2d_naive(img, np.ones((3, 3)), stride=s, padding=p)
    print(f"ảnh 28×28, kernel 3×3, stride {s}, padding {p} → {out.shape} (công thức: {(28 + 2 * p - 3) // s + 1})")

Mỗi kernel là một **bộ dò đặc trưng**: Sobel phản ứng với cạnh theo một hướng, Laplacian với mọi biên. CNN **học** các kernel như vậy từ dữ liệu thay vì thiết kế tay (bài 02 trực quan hoá kernel học được).

## 3. Nhiều kênh vào, nhiều kênh ra

Ảnh màu có $C_{in}$ kênh (RGB = 3); một lớp tích chập có $C_{out}$ bộ lọc, mỗi bộ lọc có kích thước $C_{in} \times k \times k$ (cộng qua mọi kênh vào):
$$Y[o, i, j] = b_o + \sum_{c=1}^{C_{in}}\sum_{a, b} X[c, i + a, j + b]\, K[o, c, a, b]$$
Số tham số: $C_{out}(C_{in}k^2 + 1)$ — **không phụ thuộc kích thước ảnh**.

### im2col: tích chập = nhân ma trận

Trải mỗi "mảnh" $C_{in} \times k \times k$ thành một cột → ma trận mảnh; kernel thành ma trận $C_{out} \times C_{in}k^2$ → một phép nhân ma trận lớn (GPU rất giỏi việc này).

In [ ]:
def im2col(x, kh, kw, stride=1, padding=0):
    """x: (C, H, W) → (C·kh·kw, H_out·W_out)."""
    C, H, W = x.shape
    x = np.pad(x, ((0, 0), (padding, padding), (padding, padding)))
    H_out = (H + 2 * padding - kh) // stride + 1
    W_out = (W + 2 * padding - kw) // stride + 1
    cols = np.empty((C, kh, kw, H_out, W_out))
    for a in range(kh):
        for b in range(kw):
            cols[:, a, b] = x[:, a:a + stride * H_out:stride, b:b + stride * W_out:stride]
    return cols.reshape(C * kh * kw, H_out * W_out), H_out, W_out


def conv2d_im2col(x, weight, bias, stride=1, padding=0):
    """x: (C_in, H, W), weight: (C_out, C_in, kh, kw) → (C_out, H_out, W_out)."""
    C_out, _, kh, kw = weight.shape
    cols, H_out, W_out = im2col(x, kh, kw, stride, padding)
    out = weight.reshape(C_out, -1) @ cols + bias[:, None]
    return out.reshape(C_out, H_out, W_out)


rng = np.random.default_rng(0)
x3 = rng.normal(size=(3, 28, 28))
w3, b3 = rng.normal(size=(8, 3, 5, 5)), rng.normal(size=8)
ours = conv2d_im2col(x3, w3, b3, stride=2, padding=2)
ref = F.conv2d(torch.tensor(x3[None]), torch.tensor(w3), torch.tensor(b3), stride=2, padding=2)[0].numpy()
print(f"đầu ra {ours.shape} | sai lệch lớn nhất so với F.conv2d = {np.abs(ours - ref).max():.2e}")
print(f"số tham số: {w3.size + b3.size} = C_out·(C_in·k² + 1) = {8 * (3 * 25 + 1)}")

t0 = time.perf_counter()
for _ in range(20):
    conv2d_naive(img, np.ones((3, 3)))
t_naive = (time.perf_counter() - t0) / 20
t0 = time.perf_counter()
for _ in range(20):
    conv2d_im2col(img[None], np.ones((1, 1, 3, 3)), np.zeros(1))
t_col = (time.perf_counter() - t0) / 20
print(f"một kênh 28×28: vòng lặp {t_naive * 1e3:.2f} ms | im2col {t_col * 1e3:.3f} ms")

## 4. Pooling

**Max pooling** 2×2, stride 2: giữ giá trị lớn nhất mỗi ô → giảm kích thước 4 lần, giữ "có đặc trưng ở gần đây không" — tạo chút **bất biến** với dịch chuyển nhỏ. **Average pooling** lấy trung bình. **Global average pooling** (GAP) lấy trung bình **toàn bộ** mỗi bản đồ đặc trưng → một số mỗi kênh (bài 02 cho thấy vai trò quan trọng của nó).

In [ ]:
def max_pool2d(x, size=2, stride=2):
    H_out, W_out = (x.shape[0] - size) // stride + 1, (x.shape[1] - size) // stride + 1
    out = np.empty((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            out[i, j] = x[i * stride:i * stride + size, j * stride:j * stride + size].max()
    return out


edges = np.abs(conv2d_naive(img, kernels["Laplacian (mọi cạnh)"], padding=1))
pooled = max_pool2d(edges)
ref_pool = F.max_pool2d(torch.tensor(edges[None, None]), 2)[0, 0].numpy()
print(f"{edges.shape} → max pool → {pooled.shape} | khớp F.max_pool2d: {np.allclose(pooled, ref_pool)}")
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (title, a) in zip(axes, [("ảnh", img), ("|Laplacian|", edges), ("max pool 2×2", pooled)]):
    ax.imshow(a, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()

## 5. Trường tiếp nhận (receptive field)

Mỗi đơn vị ở lớp sâu "nhìn thấy" một vùng ngày càng lớn của ảnh gốc. Với chuỗi lớp (kernel $k_l$, stride $s_l$): $r_l = r_{l-1} + (k_l - 1)\prod_{i<l}s_i$, $r_0 = 1$.

In [ ]:
def receptive_field(layers):
    r, jump = 1, 1
    out = []
    for name, k, s in layers:
        r += (k - 1) * jump
        jump *= s
        out.append((name, r))
    return out


arch = [("conv3", 3, 1), ("conv3", 3, 1), ("pool2", 2, 2), ("conv3", 3, 1), ("conv3", 3, 1), ("pool2", 2, 2), ("conv3", 3, 1)]
for name, r in receptive_field(arch):
    print(f"sau {name}: trường tiếp nhận {r}×{r}")

Hai lớp 3×3 có trường tiếp nhận bằng một lớp 5×5 nhưng ít tham số hơn ($2 \cdot 9C^2$ so với $25C^2$) và thêm một phi tuyến — ý tưởng của VGG (bài 03). Pooling/stride làm trường tiếp nhận tăng nhanh.

## 6. Đồng biến với phép dịch

Tích chập **đồng biến** (equivariant): dịch ảnh đầu vào → bản đồ đặc trưng dịch theo đúng như vậy. Kiểm chứng:

In [ ]:
shifted = np.roll(img, shift=(3, 5), axis=(0, 1))                       # dịch 3 hàng, 5 cột
k = kernels["Sobel ngang (cạnh dọc)"]
a = conv2d_naive(img, k, padding=1)
b = conv2d_naive(shifted, k, padding=1)
print(f"conv(dịch(ảnh)) == dịch(conv(ảnh)) (bỏ viền): {np.allclose(np.roll(a, (3, 5), (0, 1))[4:-4, 6:-6], b[4:-4, 6:-6])}")
print(f"tổng |đặc trưng| (sau GAP-kiểu tổng) gần như không đổi: {np.abs(a).sum():.2f} vs {np.abs(b).sum():.2f}")

- **Đồng biến** (conv): vị trí đặc trưng dịch theo vị trí vật.
- **Bất biến** (invariance): đầu ra cuối không đổi khi vật dịch — có được nhờ **pooling**, đặc biệt **global pooling** ở cuối mạng.

Đây là **thiên kiến quy nạp** (inductive bias) của CNN: "đặc trưng cục bộ, cùng một kiểu đặc trưng có thể xuất hiện ở bất kỳ đâu". Bài 02 đo lợi ích của nó.

## 🧠 Kiểm tra nhanh

1. Vì sao MLP tốn tham số và kém khi áp dụng thẳng cho ảnh?
2. Tính kích thước đầu ra: ảnh 32×32, kernel 5×5, stride 1, padding 0? Stride 2, padding 2?
3. Số tham số của `Conv2d(64, 128, kernel_size=3)`?
4. im2col biến tích chập thành phép toán gì? Vì sao nhanh?
5. Phân biệt đồng biến và bất biến với phép dịch.

## 📌 Tóm tắt

- Tích chập: bộ lọc nhỏ chia sẻ trọng số, nhìn cục bộ; $H_{out} = \lfloor(H + 2p - k)/s\rfloor + 1$; tham số $C_{out}(C_{in}k^2 + 1)$.
- im2col → nhân ma trận; pooling giảm kích thước, tăng trường tiếp nhận.
- Conv đồng biến với dịch; pooling (toàn cục) cho bất biến.

## 📚 Tài liệu tham khảo

- CS231n — *Convolutional Neural Networks* (cs231n.github.io/convolutional-networks)
- Dumoulin & Visin (2016) — *A guide to convolution arithmetic for deep learning*
- Goodfellow, Bengio & Courville (2016) — *Deep Learning*, ch. 9